# 🏨 Hotel Reservation System 7: Merging Answers and Conversation Memory

In `06_hotel_parallel_reservation_agents.ipynb` the merge node only stapled the two hotels' answers together.
Here the agents' outputs become input for another agent: an **LLM merger** reads both answers and writes one
reply, preferring hotels by **priority**. Every turn adds several agent messages to the conversation, so the
second half keeps it small: a node **summarizes** old messages into state and deletes them.

## Learning Objectives
1. **LLM merging with priorities** - The merge node hands both answers, in priority order, to a merging LLM
2. **Reading the conversation state** - Who said what, including the agents' messages to each other
3. **Summarization with `RemoveMessage`** - A running `summary` in state and old messages removed by id
4. **A changed graph on the same thread** - A shared checkpointer lets a finished thread continue on a graph
   with a summarizer at `START`

## Where this fits
`06_hotel_parallel_reservation_agents.ipynb` built the parallel two-hotel system; this notebook imports its
pieces from `hotel_common`, changes only the merge node, then adds memory management. Trimming, the cheaper
option that only shrinks what the model sees, is taught in module 02's
[`02_memory_optimizations.ipynb`](../../m23_conversational_memory_rag/notebooks/06b_memory_optimizations.ipynb) (Part 3, trim on read), so it is not repeated here.
`08_hotel_guest_language_and_sentiment.ipynb` then adds the guest's language and sentiment.

## Prerequisites
- Trimming and summarization memory from module 02's `02_memory_optimizations.ipynb` (reading only)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

---
## 📦 Step 1: Environment Setup

The setup cell loads `OPENAI_API_KEY` and imports the building blocks and `hotel_common`.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from functools import partial
from typing import Annotated, Literal, TypedDict

from langchain_core.messages import AIMessage, AnyMessage, HumanMessage, RemoveMessage, SystemMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

from datetime import date, timedelta

import hotel_common as hc  # shared hotel building blocks, in this folder (see 01_hotel_environment_and_coordinator.ipynb)
from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: shared by all agents
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini", temperature=0)

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🏨 Part 1: Two Hotels, Two Databases (from 06)

Hotel 1 (`hotel1.db`) has rooms 101–104, Hotel 2 (`hotel2.db`) rooms 105–108, each with its own read-only SQL
tools (no `book_room`). `hc.free_rooms()` gives the correct availability straight from SQLite, to compare the
agents' answers against.

In [ ]:
# ============================================================================
# TWO HOTEL DATABASES, THEIR READ-ONLY TOOLS AND THE COMPLIANCE RETRIEVER (06's setup)
# ============================================================================
HOTELS = hc.HOTELS
db_paths = hc.init_two_hotels()
sql_tools = {hotel: hc.make_sql_tools(db_paths[hotel], can_book=False) for hotel in HOTELS}
retrieve = hc.build_compliance_retriever()
rag_tools = [retrieve]

for hotel in HOTELS:
    print(f"✅ {HOTELS[hotel]}: {db_paths[hotel]}, rooms {[room[0] for room in hc.HOTEL_ROOMS[hotel]]}, "
          f"tools {[t.name for t in sql_tools[hotel]]}")

---
## 🤖 Part 2: The Agents (from 06)

The same nodes as in notebook 06, imported from `hotel_common`: the compliance checker (`hc.make_core_nodes()`),
the mapper (`hc.reservation_mapper`), one assistant per hotel on its own message key (`hc.make_hotel_assistant()`)
and the sync nodes (`hc.branch_done`). One addition: the state has a `summary` field, and `summary_context()` puts
it in front of the coordinator's and assistants' prompts. It stays empty until Part 6.

In [ ]:
# ============================================================================
# STATE, AGENT NODES AND ROUTER
# ============================================================================
class ParallelState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]          # the guest conversation
    messages_hotel1: Annotated[list[AnyMessage], add_messages]   # Hotel 1 assistant's private tool loop
    messages_hotel2: Annotated[list[AnyMessage], add_messages]   # Hotel 2 assistant's private tool loop
    summary: str                                                 # running summary of removed messages (Part 6)


AGENTS = ["reservation_assistant", "compliance_checker"]
conversation_prompt = hc.conversation_prompt(
    AGENTS,
    extra_guidelines=hc.COMPLIANCE_FIRST + "\n"  # 03's guideline, plus:
    "- The chain has two hotels. reservation_assistant checks both of them at once.",
)
# 03's compliance checker with 04's verdict; no single reservation agent: the hotel branches replace it
_, rag_assistant, generate = hc.make_core_nodes(llm, [], rag_tools)


def summary_context(state: ParallelState) -> list:
    """The running summary as a system message (empty until Part 6 starts summarizing)."""
    summary = state.get("summary")
    return [SystemMessage(content=f"Summary of the earlier conversation with this guest:\n{summary}")] if summary else []


def conv_assistant(state: ParallelState) -> dict:
    return {"messages": [llm.invoke([conversation_prompt] + summary_context(state) + state["messages"])]}


# One assistant per hotel: its own tools and message key, and the running summary in front of its messages
hotel_assistants = {hotel: hc.make_hotel_assistant(llm, hotel, sql_tools[hotel], context=summary_context)
                    for hotel in HOTELS}


def choose_next_node(state: ParallelState) -> Literal["reservation_mapper", "compliance_checker", "__end__"]:
    target = hc.route_by_mention(state["messages"], AGENTS)  # 03's router, with its loop guard
    return "reservation_mapper" if target == "reservation_assistant" else (target or END)


print("✅ Nodes defined:", ", ".join(["conv_assistant", "rag_assistant", "generate", "reservation_mapper"]
                                    + [f"{hotel}_assistant" for hotel in HOTELS]))

---
## 🧩 Part 3: A Merge Node With Priorities

The merge node is now an agent itself. It gets the guest's request and each hotel's final answer, **listed in
priority order** (the chain prefers to fill Hotel 1 first), and writes one reply:
- if several hotels can help, recommend the highest-priority one and offer the other as an alternative
- ignore a hotel that cannot help, unless none can
- keep the facts (room numbers, prices, dates) exactly as the assistants gave them

Only the merged reply goes into `messages`. The assistants' tool calls stay in their own keys, so the guest
conversation holds no half-finished tool exchanges.

In [ ]:
# ============================================================================
# MERGE NODE WITH PRIORITIES
# ============================================================================
PRIORITY = ["hotel1", "hotel2"]  # first = preferred when more than one hotel can serve the guest

merger_prompt = SystemMessage(content="""You merge the answers of the chain's reservation assistants (one per hotel)
into ONE reply to the guest.
- The answers are listed in priority order. If more than one hotel can serve the request, recommend the first one
  that can and mention the other as an alternative.
- Ignore a hotel that cannot help (no such room, already booked), unless none can: then say so plainly.
- Keep every fact (hotel, room numbers, prices, dates) exactly as the assistants gave it; never invent one.
- Speak as the hotel chain. Do not mention assistants or priorities.""")


def merge_answers(state: ParallelState) -> dict:
    request = next(m.content for m in reversed(state["messages"]) if m.type == "human")
    answers = "\n\n".join(
        f"{rank}. {HOTELS[hotel]}:\n{state[f'messages_{hotel}'][-1].content}" for rank, hotel in enumerate(PRIORITY, 1)
    )
    merged = llm.invoke([merger_prompt, HumanMessage(content=f"Guest request: {request}\n\nAnswers, in priority order:\n\n{answers}")])
    return {"messages": [AIMessage(content=merged.content, name="merge_answers")]}


print("✅ Merge node defined. Priority:", " > ".join(HOTELS[h] for h in PRIORITY))

---
## 🏗️ Part 4: The Graph

`build_graph()` wires the same graph as notebook 06. It takes an optional summarizer, which Part 6 plugs in at
`START`. Both versions share one `InMemorySaver`, so a conversation started with one graph can be continued with the
other: a thread that has finished its turn can run on a changed graph.

In [ ]:
# ============================================================================
# GRAPH BUILDER
# ============================================================================
memory = InMemorySaver()


def build_graph(summarize_node=None, route_start=None):
    builder = StateGraph(ParallelState)
    builder.add_node("conv_assistant", conv_assistant)
    builder.add_node("compliance_checker", rag_assistant)
    builder.add_node("generate", generate)
    builder.add_node("rag_tools", ToolNode(rag_tools))
    builder.add_node("reservation_mapper", hc.reservation_mapper)
    builder.add_node("merge_answers", merge_answers)

    if summarize_node is None:
        builder.add_edge(START, "conv_assistant")
    else:
        builder.add_node("summarize_conversation", summarize_node)
        builder.add_conditional_edges(START, route_start, ["summarize_conversation", "conv_assistant"])
        builder.add_edge("summarize_conversation", "conv_assistant")

    builder.add_conditional_edges("conv_assistant", choose_next_node, ["reservation_mapper", "compliance_checker", END])
    builder.add_conditional_edges("compliance_checker", tools_condition, {"tools": "rag_tools", END: "conv_assistant"})
    builder.add_edge("rag_tools", "generate")
    builder.add_edge("generate", "conv_assistant")

    for hotel in HOTELS:
        key = f"messages_{hotel}"
        builder.add_node(f"{hotel}_assistant", hotel_assistants[hotel])
        builder.add_node(f"{hotel}_tools", ToolNode(sql_tools[hotel], messages_key=key))
        builder.add_node(f"{hotel}_done", hc.branch_done)
        builder.add_edge("reservation_mapper", f"{hotel}_assistant")
        builder.add_conditional_edges(f"{hotel}_assistant", partial(tools_condition, messages_key=key),
                                      {"tools": f"{hotel}_tools", END: f"{hotel}_done"})
        builder.add_edge(f"{hotel}_tools", f"{hotel}_assistant")
    builder.add_edge([f"{hotel}_done" for hotel in HOTELS], "merge_answers")  # join: wait for both
    builder.add_edge("merge_answers", END)
    return builder.compile(checkpointer=memory)


graph = build_graph()
print("✅ Graph compiled. Edges:")
hc.print_edges(graph)

---
## ▶️ Part 5: Two Turns, Merged by Priority

Two questions on one thread:
1. a **Deluxe** room in two months: both hotels have one free (104 and 108), so the merged reply should put
   Hotel 1's room first
2. the **Family Suite** during Bob's stay: Hotel 1 has no Family Suite and Hotel 2's is booked, so nobody can help

For each turn the code prints the ground truth and checks that the merge node ran once and, where both hotels can
help, that the priority-1 hotel's room is named first.

In [ ]:
# ============================================================================
# TWO TURNS WITH CODE-CHECKED MERGING
# ============================================================================
def priority_check(reply: str, truth: dict[str, list[int]]) -> None:
    """Where both hotels can help: is the first hotel in PRIORITY with a free room named first?"""
    offering = [hotel for hotel in PRIORITY if truth[hotel]]
    if len(offering) < 2:
        print(f"ℹ️ {len(offering)} hotel(s) can help, so there is no priority to check")
        return
    first_pos = {hotel: min((reply.find(str(r)) for r in truth[hotel] if str(r) in reply), default=-1) for hotel in offering}
    top, other = offering[0], offering[1]
    ok = first_pos[top] != -1 and (first_pos[other] == -1 or first_pos[top] < first_pos[other])
    print(f"{'✅' if ok else '❌'} {HOTELS[top]}'s room {truth[top]} is named before {HOTELS[other]}'s {truth[other]}")


THREAD = "merge-demo"
stay_in = date.today() + timedelta(days=60)
clash_in = date.today() + timedelta(days=31)  # inside Bob's Family Suite booking (today+30 to today+33)
turns = [
    ("Deluxe", stay_in, stay_in + timedelta(days=2),
     f"Hi! Is a Deluxe room available from {stay_in} to {stay_in + timedelta(days=2)}? What does it cost per night?"),
    ("Family Suite", clash_in, clash_in + timedelta(days=2),
     f"And is the Family Suite free from {clash_in} to {clash_in + timedelta(days=2)}?"),
]

for room_type, start, end, text in turns:
    nodes_run, state = hc.run_traced(graph, text, thread_id=THREAD)
    reply = state["messages"][-1]
    truth = {hotel: hc.free_rooms(db_paths[hotel], room_type, start, end) for hotel in HOTELS}
    print(f"🤵 Reply ({reply.name or 'conv_assistant'}):\n{reply.content}\n")
    print(f"Ground truth, free {room_type} rooms: " + ", ".join(f"{HOTELS[h]} {truth[h]}" for h in PRIORITY))
    merge_runs = nodes_run.count("merge_answers")
    print(f"{'✅' if merge_runs == 1 else '❌'} merge node ran {merge_runs} time(s)")
    if merge_runs:
        priority_check(reply.content, truth)
    print("=" * 80)

**Reading the output:** on the Deluxe turn the ✅ means the merged reply names Hotel 1's room 104 before Hotel 2's
room 108, as the priority asks. On the Family Suite turn neither hotel can help (both ground-truth lists are empty),
so read the reply: it should say plainly that no Family Suite is free on those dates.

### Who said what
The thread's `messages` now hold the whole exchange between the agents: the guest, the coordinator forwarding
work, the compliance checker's tool call and verdict, and the merged reply. Each hotel's own tool loop is kept
separately in `messages_hotel1` / `messages_hotel2`.

In [ ]:
# ============================================================================
# THE CONVERSATION STATE, MESSAGE BY MESSAGE
# ============================================================================
def show_messages(messages) -> None:
    for i, m in enumerate(messages):
        who = m.name or {"human": "guest", "ai": "conv_assistant", "tool": "tool"}.get(m.type, m.type)
        text = " ".join(str(m.content).split())[:90]
        if getattr(m, "tool_calls", None):
            text = "calls " + ", ".join(tc["name"] for tc in m.tool_calls)
        print(f"{i:>3}  {m.type:<6} {who:<20} {text}")


config = {"configurable": {"thread_id": THREAD}}
values = graph.get_state(config).values
show_messages(values["messages"])
print(f"\n{len(values['messages'])} messages in the conversation; branch keys hold "
      + ", ".join(f"{hotel}: {len(values[f'messages_{hotel}'])}" for hotel in HOTELS))

---
## 🗜️ Part 6: Summarizing Into State

Trimming (`trim_messages`, module 02) only shrinks what the model sees: whatever falls outside the window is
forgotten. Summarizing keeps the gist of it. A `summarize_conversation` node runs at `START` when the
stored conversation is longer than `SUMMARIZE_AFTER` messages:
1. it keeps the last `KEEP_TURNS` guest turns (the new message is one of them); everything before is "old"
2. it asks the LLM to fold the old messages into the running summary (`summary` in state)
3. it returns `RemoveMessage(id=...)` for each old message, so `add_messages` deletes them from the state

The cut is always at a guest message, so no tool call is separated from its result. The coordinator and the
assistants see the summary through `summary_context()` (Part 2); the reply to the guest is not touched.
`needs_summary()` is a pure routing function on the message count.

The new graph continues the **same thread** as Part 5, through the shared checkpointer.

In [ ]:
# ============================================================================
# SUMMARIZER NODE AND THE NEW GRAPH
# ============================================================================
SUMMARIZE_AFTER = 6  # summarize when the stored conversation has more messages than this
KEEP_TURNS = 1       # most recent guest turns kept word for word (the new message counts as one)


def needs_summary(state: ParallelState) -> Literal["summarize_conversation", "conv_assistant"]:
    return "summarize_conversation" if len(state["messages"]) > SUMMARIZE_AFTER else "conv_assistant"


def transcript(messages) -> str:
    lines = []
    for m in messages:
        if m.type == "human":
            lines.append(f"Guest: {m.content}")
        elif m.type == "ai" and m.content and not m.tool_calls:
            lines.append(f"{m.name or 'Coordinator'}: {m.content}")
    return "\n".join(lines)


def summarize_conversation(state: ParallelState) -> dict:
    messages = state["messages"]
    guest_turns = [i for i, m in enumerate(messages) if m.type == "human"]
    cut = guest_turns[-KEEP_TURNS] if len(guest_turns) >= KEEP_TURNS else 0
    old = messages[:cut]
    if not old:
        return {}
    summary = llm.invoke([
        SystemMessage(content="You keep a running summary of a hotel guest's conversation with the hotel's support "
                      "system. Extend the existing summary with the new lines. Keep every concrete fact: names, room "
                      "types, room numbers, prices, dates, what was offered and what was refused. Reply with the "
                      "summary only."),
        HumanMessage(content=f"Existing summary:\n{state.get('summary') or '(none)'}\n\nNew lines:\n{transcript(old)}"),
    ]).content
    return {"summary": summary, "messages": [RemoveMessage(id=m.id) for m in old]}


summarizing_graph = build_graph(summarize_node=summarize_conversation, route_start=needs_summary)
print("✅ Graph with summarizer compiled. Edges from START:")
for edge in summarizing_graph.get_graph().edges:
    if edge.source == "__start__" or edge.source == "summarize_conversation":
        print(f"  {edge.source} -> {edge.target}")

A third question on the same thread refers back to the first turn. Before the turn the thread holds the messages
of Part 5, more than `SUMMARIZE_AFTER`, so the summarizer should run first. The checks compare the stored
conversation before and after the turn.

In [ ]:
# ============================================================================
# THIRD TURN: SUMMARIZE, REMOVE BY ID, THEN ANSWER
# ============================================================================
before = summarizing_graph.get_state(config).values["messages"]
old_ids = {m.id for m in before}

nodes_run, state = hc.run_traced(summarizing_graph, "Remind me: which Deluxe room did you recommend, and at what price?",
                              thread_id=THREAD)
after = state["messages"]
reply = after[-1].content
summary = state.get("summary", "")

print("Nodes, in the order they ran:\n  " + " → ".join(nodes_run) + "\n")
print(f"🤵 Reply:\n{reply}\n")
print(f"🗒️ Summary stored in state:\n{summary}\n")
print("Conversation now:")
show_messages(after)

print("\n--- Checks ---")
print(f"{'✅' if 'summarize_conversation' in nodes_run else '❌'} summarize_conversation ran")
print(f"   messages before the turn: {len(before)} | after the turn: {len(after)}")
left = old_ids & {m.id for m in after}
print(f"{'✅' if not left else '❌'} all {len(old_ids)} earlier messages removed by id ({len(left)} left)")
print(f"{'✅' if summary else '❌'} summary kept in state['summary'] ({len(summary)} characters)")
print(f"{'✅' if 'Deluxe' in summary else '❌'} the summary still mentions the Deluxe offer")
print(f"{'✅' if summary and summary not in reply else '❌'} the summary is not pasted into the guest's reply")

**Reading the output:** the conversation shrank from the Part 5 messages to just this turn's, and none of the old
message ids are left; their content lives on in `summary`. The reply should still name the Deluxe room from the
first turn, which the model can only know from the summary. On the next long turn the summarizer extends the same
summary rather than starting over.

To chat with the system yourself, run `hc.chat_with_agent(summarizing_graph, "my-thread")` in a new cell.

---
## 📝 Summary

### 1. LLM merging with priorities
- An agent's output can be another agent's input: the **merge node** reads both hotels' answers, in
  **priority order**, and writes one reply. Only that reply enters the guest conversation.

### 2. Reading the conversation state
- `graph.get_state(config).values` shows the whole agent-to-agent exchange, message by message.

### 3. Summarization with `RemoveMessage`
- **Summarizing** folds old messages into a `summary` key and deletes them with `RemoveMessage(id=...)`, cutting
  at a guest message; the summary feeds the prompts, never the reply. (`trim_messages`, the view-only
  alternative, is in module 02's `02_memory_optimizations.ipynb`.)

### 4. A changed graph on the same thread
- A thread that finished its turn can continue on a **changed graph** with the same checkpointer.

### Next Steps
- `08_hotel_guest_language_and_sentiment.ipynb`: the system answers guests in their own language and calms
  angry ones